In [1]:
import re
from datasets import load_dataset
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

In [21]:
SEQ_LEN = 100

In [2]:
def download_dataset():
    story_dataset = load_dataset("cfahlgren1/tinystories-gpt4-clean")
    print(story_dataset)
    print(story_dataset["train"][0])
    return story_dataset

story_dataset = download_dataset()

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

tinystories_gpt4_clean.parquet: reconstructing file:   0%|          |  0.00B /  673MB            

tinystories_gpt4_clean.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2732634 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2732634
    })
})
{'text': 'Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.\nJack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.\nOnce the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!\nThe more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that.'}


In [22]:
def save_stories_to_file(story_dataset, story_file_path):
    Path("data").mkdir(exist_ok=True)

    with story_file_path.open("w", encoding="utf-8") as story_file:
        for story_number, story_item in enumerate(story_dataset["train"]):
            story_text = story_item["text"]
            story_file.write(story_text)
            story_file.write("\n<END_STORY>\n")

            if (story_number + 1) % 100000 == 0:
                print(f"Processed {story_number + 1} stories")

    print(f"Done! Text saved to: {story_file_path}")

story_file_path = Path("data/tinystories_all.txt")
save_stories_to_file(story_dataset, story_file_path)

Processed 100000 stories
Processed 200000 stories
Processed 300000 stories
Processed 400000 stories
Processed 500000 stories
Processed 600000 stories
Processed 700000 stories
Processed 800000 stories
Processed 900000 stories
Processed 1000000 stories
Processed 1100000 stories
Processed 1200000 stories
Processed 1300000 stories
Processed 1400000 stories
Processed 1500000 stories
Processed 1600000 stories
Processed 1700000 stories
Processed 1800000 stories
Processed 1900000 stories
Processed 2000000 stories
Processed 2100000 stories
Processed 2200000 stories
Processed 2300000 stories
Processed 2400000 stories
Processed 2500000 stories
Processed 2600000 stories
Processed 2700000 stories
Done! Text saved to: data/tinystories_all.txt


In [23]:
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as story_file:
        return story_file.read()

all_story_text = read_text_file("data/tinystories_all.txt")

In [24]:
raw_words = []
raw_words.extend(all_story_text[:1999999].split(" "))
raw_words.append("\n")

In [25]:
def split_punctuation(raw_words):
    split_tokens = []
    for raw_word in raw_words:
        raw_word = raw_word.lower()
        raw_word = re.sub(r"([^a-z0-9\s])", r" \1 ", raw_word)
        split_tokens.extend(raw_word.split())
    return split_tokens

token_list = split_punctuation(raw_words)

In [26]:
def keep_only_letters(token_list):
    letters_only_words = []
    for token in token_list:
        token = token.lower()
        token = re.sub(r"[^a-z]", "", token)
        if token:
            letters_only_words.append(token)
    return letters_only_words

letters_only_words = keep_only_letters(token_list)

In [27]:
unique_sorted_words = sorted(list(set(letters_only_words)))
unique_words_string = " ".join(unique_sorted_words)
print(len(unique_words_string))
unique_words_string

34574


'a able about above accept accepted accident accidentally accidently accidents accomplished ached achieve achieved achievements achoo acorn across act acted acting action actions activity actors actually add added adding adds admire admired admiring adorable adult adults advantage adventure adventures adventurous advice aeroplane afar afloat afraid after afternoon again against age ago agree agreed agreement agrees ah ahead ahh ahhh ahoy aid aiden aim aimed aims air airplane airport aisle alarm alert alex alice alien alive all allergy alligator allow allowed allowing almost alone along alongside already alright also although always am amanda amazed amazement amazing ambitions ambitious ambulance amongst amount amusement amy an anchor ancient and andvery andy angel angels angles angry animal animals ann anna annabelle annie annoyed another answer answered answers ant anthony ants anxious any anymore anyone anything anytime anyway anywhere apart apologised apologize apologized appear app

In [28]:
def build_probability_matrix(token_list):
    vocab_list = list(dict.fromkeys(token_list))
    vocab_size = len(vocab_list)
    word_counts = [token_list.count(word) for word in vocab_list]

    probability_matrix = np.zeros((vocab_size, vocab_size))

    for position in range(len(token_list) - 1):
        current_index = vocab_list.index(token_list[position])
        next_index = vocab_list.index(token_list[position + 1])
        probability_matrix[current_index][next_index] += 1

    for row in range(vocab_size):
        probability_matrix[row] = probability_matrix[row] / word_counts[row]

    return vocab_list, vocab_size, word_counts, probability_matrix

vocab_list, vocab_size, word_counts, probability_matrix = build_probability_matrix(token_list)

In [29]:
probability_table = pd.DataFrame(probability_matrix, index=vocab_list, columns=vocab_list)
probability_table.head(10)

,once,there,was,a,little,boy,named,jack,.,he,...,exhausted,proudest,lung,staff,villagers,politeest,brand,energized,shares,su
once,0.000000,0.044994,0.000634,0.000000,0.000000,0.000000,0.000000,0.000000,0.001267,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
there,0.000487,0.000000,0.737329,0.000487,0.000000,0.000000,0.000000,0.000000,0.044834,0.000487,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
was,0.000213,0.001807,0.000000,0.222045,0.000000,0.000000,0.000213,0.000213,0.006909,0.000000,...,0.000106,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
a,0.000000,0.000000,0.000000,0.000000,0.096648,0.013248,0.000000,0.000000,0.000217,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000072,0.0,0.0,0.0
little,0.000000,0.000000,0.000000,0.000000,0.000000,0.217586,0.000000,0.000497,0.001490,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
boy,0.000000,0.000000,0.065050,0.000000,0.000000,0.000000,0.566703,0.000000,0.066152,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
named,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004692,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
jack,0.000000,0.000000,0.190789,0.013158,0.000000,0.000000,0.000000,0.000000,0.085526,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
.,0.000150,0.001800,0.000000,0.004874,0.000150,0.000000,0.000000,0.001325,0.000200,0.109448,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
he,0.000000,0.000000,0.108177,0.000000,0.000000,0.000000,0.000402,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


In [30]:
probability_table["boy"]["little"]

np.float64(0.21758569299552907)

In [31]:
def build_word_to_index(token_list):
    word_to_index = {}
    for index, word in enumerate(sorted(list(set(token_list)))):
        word_to_index[word] = index
    return word_to_index

word_to_index = build_word_to_index(token_list)
word_to_index["a"]

23

In [32]:
def build_index_to_word(token_list):
    index_to_word = {}
    for index, word in enumerate(sorted(list(set(token_list)))):
        index_to_word[index] = word
    return index_to_word

index_to_word = build_index_to_word(token_list)
index_to_word[4]

'-'

In [33]:
def data_and_labels():
    xs = []
    ys = []
    for i in range(len(token_list) - SEQ_LEN):
        xs.append(token_list[i:i+SEQ_LEN])
        ys.append(token_list[i+SEQ_LEN])
    return xs, ys

xs, ys = data_and_labels()

In [34]:
xs[10],[ys[10]]

(['was',
  'only',
  'three',
  'years',
  'old',
  'and',
  'had',
  'lots',
  'of',
  'things',
  'he',
  'wanted',
  'to',
  'do',
  '.',
  'one',
  'day',
  'he',
  'saw',
  'something',
  'very',
  'special',
  'in',
  'the',
  'park',
  '-',
  'a',
  'big',
  'wheel',
  '!',
  'it',
  'was',
  'big',
  'and',
  'bright',
  'and',
  'looked',
  'very',
  'inviting',
  '.',
  'jack',
  'wanted',
  'to',
  'get',
  'on',
  'the',
  'wheel',
  ',',
  'so',
  'he',
  'ran',
  'to',
  'it',
  '.',
  'he',
  'put',
  'his',
  'hands',
  'on',
  'it',
  'and',
  'gently',
  'started',
  'turning',
  'it',
  'around',
  '.',
  'the',
  'wheel',
  'spun',
  'faster',
  'and',
  'he',
  'laughed',
  'as',
  'he',
  'felt',
  'the',
  'wind',
  'blowing',
  'in',
  'his',
  'face',
  '.',
  'once',
  'the',
  'wheel',
  'slowed',
  'down',
  ',',
  'jack',
  'got',
  'off',
  'and',
  'looked',
  'around',
  '.',
  'he',
  'had',
  'such'],
 ['a'])

In [35]:
for i in range(len(ys)):
    xs[i] = [word_to_index[word] for word in xs[i]]
    ys[i] = word_to_index[ys[i]]


In [36]:
xs[1],[ys[1]]

([4313,
  4704,
  23,
  2341,
  477,
  2647,
  2086,
  5,
  1844,
  4704,
  2777,
  4339,
  4901,
  2767,
  128,
  1791,
  2377,
  2754,
  4323,
  1844,
  4692,
  4393,
  1150,
  5,
  2775,
  1039,
  1844,
  3527,
  3894,
  4630,
  3933,
  2029,
  4305,
  2871,
  4,
  23,
  368,
  4765,
  0,
  2080,
  4704,
  368,
  128,
  509,
  128,
  2366,
  4630,
  2073,
  5,
  2086,
  4692,
  4393,
  1661,
  2773,
  4305,
  4765,
  3,
  3866,
  1844,
  3251,
  4393,
  2080,
  5,
  1844,
  3195,
  1906,
  1810,
  2773,
  2080,
  128,
  1658,
  4031,
  4522,
  2080,
  185,
  5,
  4305,
  4765,
  3986,
  1419,
  128,
  1844,
  2238,
  192,
  1844,
  1442,
  4305,
  4808,
  417,
  2029,
  1906,
  1391,
  5,
  2774,
  4305,
  4765,
  3805,
  1184,
  3,
  2086],
 [1720])

In [37]:
xs = np.array(xs)
ys = np.array(ys)
vocab_size = len(word_to_index)
print(xs.shape, ys.shape)

(490896, 100) (490896,)


In [38]:
from tensorflow.keras import layers

class WordAndPosition(layers.Layer):
    def __init__(self, vocab_size, size):
        super().__init__()
        self.word_layer = layers.Embedding(vocab_size, size)
        self.position_layer = layers.Embedding(SEQ_LEN, size)

    def call(self, x):
        positions = tf.range(SEQ_LEN)
        return self.word_layer(x) + self.position_layer(positions)

def build_model(vocab_size, size=64, blocks=2):
    model_input = layers.Input((SEQ_LEN,), dtype="int32")
    x = WordAndPosition(vocab_size, size)(model_input)

    for _ in range(blocks):
        look_back = layers.MultiHeadAttention(4, size // 4)(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + look_back)
        think = layers.Dense(size * 4, activation="relu")(x)
        think = layers.Dense(size)(think)
        x = layers.LayerNormalization()(x + think)

    last_word = layers.Lambda(lambda t: t[:, -1, :])(x)
    model_output = layers.Dense(vocab_size)(last_word)
    return tf.keras.Model(model_input, model_output)

gpt_model = build_model(vocab_size)
gpt_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
gpt_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_and_position_1 │ (None, 100, 64)   │    323,072 │ input_layer_1[0]… │
│ (WordAndPosition)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 64)   │     16,640 │ word_and_positio… │
│ (MultiHeadAttentio… │                   │            │ word_and_positio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 100, 64)   │          0 │ word_and_positio… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 100, 64)   │        128 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 100, 256)  │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 100, 64)   │     16,448 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 100, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 100, 64)   │        128 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 64)   │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 100, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 100, 64)   │        128 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 100, 256)  │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 100, 64)   │     16,448 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 100, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 100, 64)   │        128 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 64)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 4948)      │    321,620 │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 744,660 (2.84 MB)

 Trainable params: 744,660 (2.84 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
gpt_model.fit(xs, ys, batch_size=64, epochs=20)

Epoch 1/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 85s 10ms/step - loss: 3.5072
Epoch 2/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 68s 9ms/step - loss: 2.8511
Epoch 3/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 61s 8ms/step - loss: 2.6283
Epoch 4/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.4883
Epoch 5/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.3874
Epoch 6/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.3110
Epoch 7/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.2485
Epoch 8/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.1989
Epoch 9/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.1556
Epoch 10/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.1179
Epoch 11/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.0830
Epoch 12/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.0551
Epoch 13/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.0283
Epoch 14/20
7671/7671 ━━━━━━━━━━━━━━━━━━━━ 56s 7ms/step - loss: 2.0033
Epoch 15/20
76

In [41]:
def write_story(start_numbers, pad_size=0, how_many_words=30):
    numbers = list(start_numbers)

    for _ in range(how_many_words):
        last_words = np.array([numbers[-SEQ_LEN:]])
        scores = gpt_model.predict(last_words, verbose=0)[0]
        chances = np.exp(scores - scores.max())
        chances = chances / chances.sum()
        numbers.append(np.random.choice(len(chances), p=chances))

    return " ".join(index_to_word[int(n)] for n in numbers[pad_size:])


def prompt(text="once there was a little boy named jack . he", how_many_words=500):
    words = text.lower().split()

    unknown_words = [word for word in words if word not in word_to_index]
    if unknown_words:
        print("Words the model does not know:", unknown_words)
        words = [word for word in words if word in word_to_index]

    start_numbers = [word_to_index[word] for word in words]
    start_numbers = start_numbers[-SEQ_LEN:]

    pad_size = SEQ_LEN - len(start_numbers)
    if pad_size > 0:
        random_window = xs[np.random.randint(len(xs))]
        filler = [int(n) for n in random_window[:pad_size]]
        start_numbers = filler + start_numbers

    return write_story(start_numbers, pad_size, how_many_words)


prompt("what is a boy and a girl doing in the story ?")

'what is a boy and a girl doing in the story ? one day , go of a nice girl named lily came over to mom and asked mom . lily was very obedient and her mom . her mom was always talked to her when they woke up , it was icy because lily liked her busy work . one day , lily felt like a little girl named sue found a tall leaf . sue loved the leaf very much . kitty wanted to eat the leaf too . amy went to the post and her mom said , " why can help you want to climb it ? let \' s eat my leaf too . they went to the river to help kitty feel better . they would swim up and down . but then , the dog got so far away that . fluffy \' s friends were so happy and decided to share the leaf together . they found the magic leaf . even though it was not good at found the leaf right . < end _ story > once upon a time , there was a farm . the farmer was rich and wanted to be kind and gentle named tom . lily loved to run and fly , but lily didn \' t quit playing in . one day , lily \' s hay was the best . li

In [43]:
gpt_model.save("gpt_model.keras")
gpt_model.save("gpt_model.h5")
